# CosMx → Celldega pre-processing (standalone)

Generates Celldega **DegaFiles** (LandscapeFiles) for a CosMx SMI dataset by **converting CosMx flat files into the Xenium file format** and then reusing the existing `celldega.pre` Xenium pre-processing functions (no changes to the `pre` module).

**Dataset:** colon CRC discovery sample `S0` (NanoString WTx, 18,935 genes, ~494k cells, ~1.12B transcripts). Morphology images come from the OME-ZARR inside `Napari.zip`.

### Key facts / decisions
* **Coordinate frame:** CosMx `*_global_px` == Napari image level-0 pixels (verified: flat-file px span 114925×120960 ≈ image 114908×120942). So cells/transcripts need **no** coordinate transform to align with the morphology image.
* **Polygons fix:** CosMx flat-file polygons have correct cell shape/size but wrong global position (per-FOV centroids spread 2×). We keep the shape and translate each polygon so its centroid matches the (validated) metadata centroid.
* **Image scale:** images are built at zarr **level 1** (half-res, 0.24 µm/px). We map `global_px → level-1 px` by baking a `0.5` scale into the transform matrix (NOT via `image_scale`, which celldega applies inconsistently across functions, so keep it = 1).
* **Transcripts:** `make_trx_tiles_row_groups` loads the whole transcript parquet into RAM; 1.12B rows would OOM, so we subsample to ~250M for tiling (still very dense).
* Run everything on fast local disk; copy DegaFiles to the (slow s3fs) workbench afterwards. `use_row_groups=True` keeps the file count low for a fast copy.

**Requirements:** `celldega` (v0.18.0), `pyvips` (with system libvips), `pyarrow`, `polars`, `scanpy`, `scipy`, `zarr`.

Edit the paths/constants at the top of each step for your environment.

## Step 1 — metadata + polygons → Xenium `cells.csv.gz` and `cell_boundaries.parquet`
Run the final `gzip` to produce `cells.csv.gz` after this cell (the cell writes `cells.csv`).

In [ ]:
"""Convert CosMx metadata + polygons -> Xenium-format cells.csv.gz and cell_boundaries.parquet.

CosMx flat-file polygons have the CORRECT cell shape/size (raw bbox == metadata Width/Height,
ratio 1.00) but their global POSITION is wrong: within each FOV the per-cell centroids are spread
~2x relative to the true (metadata/transcript/image) frame. So we keep the polygon shape and
translate each cell's polygon so its centroid lands on the validated metadata centroid (which is
confirmed to align with the morphology image):
    vertex_fixed = raw_vertex - mean(raw_vertices_of_cell) + metadata_centroid_of_cell
"""
import polars as pl
from pathlib import Path

COSMX = Path("/home/jovyan/local/cosmx_data")
OUT = Path("/home/jovyan/local/cosmx_xenium/S0")
OUT.mkdir(parents=True, exist_ok=True)

# ---- cells.csv.gz (Xenium: cell_id, x_centroid, y_centroid) ----
meta = pl.read_csv(
    COSMX / "S0_metadata_file.csv.gz",
    columns=["cell", "CenterX_global_px", "CenterY_global_px"],
    schema_overrides={"CenterX_global_px": pl.Float64, "CenterY_global_px": pl.Float64},
)
cells = meta.rename(
    {"cell": "cell_id", "CenterX_global_px": "x_centroid", "CenterY_global_px": "y_centroid"}
).select(["cell_id", "x_centroid", "y_centroid"])
print("cells:", cells.shape)
cells.write_csv(OUT / "cells.csv")  # gzip after with shell

# ---- cell_boundaries.parquet (Xenium: cell_id, vertex_x, vertex_y) ----
# Keep polygon shape, translate so each cell's polygon centroid == metadata centroid.
poly = pl.read_csv(
    COSMX / "S0-polygons.csv.gz",
    columns=["cell", "x_global_px", "y_global_px"],
    schema_overrides={"x_global_px": pl.Float64, "y_global_px": pl.Float64},
).rename({"cell": "cell_id"})
# per-cell raw polygon centroid (mean of vertices)
raw_c = poly.group_by("cell_id").agg([
    pl.col("x_global_px").mean().alias("rcx"),
    pl.col("y_global_px").mean().alias("rcy"),
])
poly = poly.join(raw_c, on="cell_id", how="left").join(
    cells.rename({"x_centroid": "mcx", "y_centroid": "mcy"}), on="cell_id", how="inner"
)
bounds = poly.with_columns([
    (pl.col("x_global_px") - pl.col("rcx") + pl.col("mcx")).alias("vertex_x"),
    (pl.col("y_global_px") - pl.col("rcy") + pl.col("mcy")).alias("vertex_y"),
]).select(["cell_id", "vertex_x", "vertex_y"])
# drop degenerate polygons (<4 vertices) — shapely LinearRing needs >=4 coordinates
_keep = bounds.group_by("cell_id").len().filter(pl.col("len") >= 4).select("cell_id")
bounds = bounds.join(_keep, on="cell_id", how="inner")
print("boundaries rows:", bounds.shape, "unique cells:", bounds["cell_id"].n_unique())
bounds.write_parquet(OUT / "cell_boundaries.parquet")
print("DONE cells+boundaries")


In [ ]:
import subprocess
subprocess.run(['gzip', '-f', '/home/jovyan/local/cosmx_xenium/S0/cells.csv'], check=True)
print('cells.csv.gz written')

## Step 2 — expression matrix → 10x MTX `cell_feature_matrix/`
pyarrow streaming, parses only real genes (drops `Negative*` / `SystemControl*`). ~8 min; the final `mmwrite` of ~550M nonzeros dominates.

In [ ]:
"""Convert CosMx exprMat CSV -> 10x MTX (cell_feature_matrix/) using pyarrow streaming (fast).

Drops control columns (Negative*, SystemControl*). pyarrow parses only the kept columns.
Writes matrix.mtx.gz (genes x cells), barcodes.tsv.gz, features.tsv.gz so read_cbg_mtx works
(read_cbg_mtx transposes -> cells x genes indexed by barcodes, cols = features[1]).
"""
import gzip, shutil, time
from pathlib import Path
import numpy as np
import pyarrow as pa
import pyarrow.csv as pacsv
import scipy.sparse as sp
from scipy.io import mmwrite

COSMX = "/home/jovyan/local/cosmx_data/S0_exprMat_file.csv.gz"
OUTDIR = Path("/home/jovyan/local/cosmx_xenium/S0/cell_feature_matrix")
OUTDIR.mkdir(parents=True, exist_ok=True)
SLIDE = 1

# header
with gzip.open(COSMX, "rt") as f:
    header = f.readline().strip().split(",")
genes = [g for g in header[2:]
         if not (g.startswith("Negative") or g.startswith("SystemControl"))]
print(f"kept genes {len(genes)} of {len(header)-2}")

include = ["fov", "cell_ID"] + genes
col_types = {"fov": pa.int32(), "cell_ID": pa.int64()}
for g in genes:
    col_types[g] = pa.int32()

read_opts = pacsv.ReadOptions(block_size=256 << 20)
conv_opts = pacsv.ConvertOptions(include_columns=include, column_types=col_types)

t = time.time()
reader = pacsv.open_csv(COSMX, read_options=read_opts, convert_options=conv_opts)
barcodes = []
blocks = []
nrows = 0
for batch in reader:
    fov = batch.column("fov").to_numpy()
    cid = batch.column("cell_ID").to_numpy()
    barcodes.extend(f"c_{SLIDE}_{f}_{c}" for f, c in zip(fov, cid))
    # gene matrix: stack the gene columns (zero-copy int32 arrays) -> dense -> csr
    mat = np.empty((batch.num_rows, len(genes)), dtype=np.int32)
    for j, g in enumerate(genes):
        mat[:, j] = batch.column(g).to_numpy(zero_copy_only=False)
    blocks.append(sp.csr_matrix(mat))
    nrows += batch.num_rows
    print(f"  {nrows} cells  {time.time()-t:.0f}s", flush=True)

cbg = sp.vstack(blocks).tocsr()
print("cbg (cells x genes):", cbg.shape, "nnz", cbg.nnz, "%.0fs" % (time.time()-t))

with gzip.open(OUTDIR / "barcodes.tsv.gz", "wt") as f:
    f.write("\n".join(barcodes) + "\n")
with gzip.open(OUTDIR / "features.tsv.gz", "wt") as f:
    f.write("\n".join(f"{g}\t{g}\tGene Expression" for g in genes) + "\n")
mtx = OUTDIR / "matrix.mtx"
mmwrite(str(mtx), cbg.T.tocoo(), field="integer")
with open(mtx, "rb") as fi, gzip.open(str(mtx) + ".gz", "wb") as fo:
    shutil.copyfileobj(fi, fo, length=16 << 20)
mtx.unlink()
print("DONE %.0fs" % (time.time()-t), sorted(p.name for p in OUTDIR.iterdir()))


## Step 3 — transcripts → Xenium `transcripts.parquet` (+ subsample)
pyarrow streams the 10.6 GB gzip (bounded memory). Then subsample to ~250M rows for tiling.

In [ ]:
"""Convert CosMx tx_file (10.6GB gz) -> Xenium-format transcripts.parquet via pyarrow streaming.

Streams the gzipped CSV batch-by-batch (bounded memory), filters control targets, renames to
Xenium columns (cell_id, transcript_id, feature_name, x_location, y_location). Coords are already
in the image frame.
"""
import time
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc
import pyarrow.parquet as pq

TX = "/home/jovyan/local/cosmx_data/S0_tx_file.csv.gz"
OUT = "/home/jovyan/local/cosmx_xenium/S0/transcripts.parquet"

read_opts = pacsv.ReadOptions(block_size=64 << 20)  # 64MB blocks
parse_opts = pacsv.ParseOptions(delimiter=",")
conv_opts = pacsv.ConvertOptions(
    column_types={"x_global_px": pa.float64(), "y_global_px": pa.float64(),
                  "fov": pa.int32(), "cell_ID": pa.int64()},
    include_columns=["cell", "target", "x_global_px", "y_global_px"],
)

out_schema = pa.schema([
    ("cell_id", pa.string()),
    ("transcript_id", pa.int64()),
    ("feature_name", pa.string()),
    ("x_location", pa.float64()),
    ("y_location", pa.float64()),
])

t = time.time()
reader = pacsv.open_csv(TX, read_options=read_opts, parse_options=parse_opts, convert_options=conv_opts)
writer = pq.ParquetWriter(OUT, out_schema, compression="zstd")
tid = 0
nrows = 0
nbatch = 0
for batch in reader:
    tgt = batch.column("target")
    is_ctrl = pc.or_(pc.starts_with(tgt, "Negative"), pc.starts_with(tgt, "SystemControl"))
    keep = pc.invert(is_ctrl)
    b = batch.filter(keep)
    n = b.num_rows
    if n == 0:
        continue
    tids = pa.array(range(tid, tid + n), type=pa.int64())
    tid += n
    out = pa.record_batch(
        [b.column("cell"), tids, b.column("target"),
         b.column("x_global_px"), b.column("y_global_px")],
        schema=out_schema,
    )
    writer.write_batch(out)
    nrows += n
    nbatch += 1
    if nbatch % 50 == 0:
        print(f"  {nrows:,} kept rows, {time.time()-t:.0f}s", flush=True)
writer.close()
print(f"DONE tx->transcripts: {nrows:,} rows in {time.time()-t:.0f}s -> {OUT}")


In [ ]:
"""Subsample transcripts.parquet (~1.12B rows) to ~250M for memory-safe tile generation."""
import time
import polars as pl

SRC = "/home/jovyan/local/cosmx_xenium/S0/transcripts.parquet"
DST = "/home/jovyan/local/cosmx_xenium/S0/transcripts_sub.parquet"
KEEP_PER_MILLE = 223  # ~22.3% -> ~250M rows

t = time.time()
(
    pl.scan_parquet(SRC)
    .filter((pl.col("transcript_id") % 1000) < KEEP_PER_MILLE)
    .sink_parquet(DST, compression="zstd")
)
n = pl.scan_parquet(DST).select(pl.len()).collect().item()
print(f"DONE subsample -> {n:,} rows in {time.time()-t:.0f}s")


## Step 4 — drive the Celldega Xenium pipeline → DegaFiles
Reuses `celldega.pre.*` (technology=`Xenium`) with `use_row_groups=True`: cell metadata, leiden clustering, meta-gene, CBG row-groups, cluster files, **image pyramids from the Napari OME-ZARR (DNA + PanCK/Membrane/CD45)**, transcript tiles, boundary tiles, landscape parameters.

In [ ]:
"""Drive the existing celldega Xenium pre-processing on CosMx data converted to Xenium format.

Reuses dega.pre.* functions (technology="Xenium") with use_row_groups=True. Does NOT modify the
pre module. Inputs are the Xenium-format intermediates in DATA_DIR; morphology images come from
the Napari OME-ZARR inside Napari.zip (channels rendered into Celldega DeepZoom pyramids).

Coordinate frame: CosMx global_px == Napari image level-0 px. We build images at zarr level 1
(half res) so IMAGE_SCALE=2 maps global_px -> level-1 px. Transform is identity.
"""
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyvips
import zarr

import celldega as dega

# ---------------- config ----------------
TECH = "Xenium"
DATA_DIR = Path("/home/jovyan/local/cosmx_xenium/S0")          # Xenium-format inputs
DEGA = Path("/home/jovyan/local/cosmx_dega_files/S0")          # output DegaFiles
DEGA.mkdir(parents=True, exist_ok=True)
TILE_SIZE = 250
# Images are built at zarr level 1 (half of full-res global_px). We map global_px -> level-1 px
# by baking a 0.5 scale into the (otherwise identity) transform matrix, NOT via image_scale:
# image_scale is applied inconsistently across celldega funcs (meta/boundary divide, trx multiply),
# so it is only safe at 1. The transform matrix is applied uniformly, so 0.5*I scales all layers.
IMAGE_SCALE = 1
COORD_SCALE = 0.25       # global_px (level 0) -> level 2 px (zarr level 2 == /4)
ZARR_LEVEL = "2"
MAX_WORKERS = 8
MRG = 400                # max row groups per file

ZIP = "/home/jovyan/local/cosmx_data/Napari.zip"
ZPFX = "Run_b8806732-4c8c-4a36-bb0f-52a601a12475/20240620_231957_S2/Napari/images"
# CosMx channel -> Celldega channel name / button / color
# Visualize DNA (nuclear/DAPI background) + 3 protein/membrane stains.
CHANNELS = [
    ("DNA", "dapi", "DNA", [0, 0, 255]),
    ("PanCK", "panck", "PanCK", [0, 255, 0]),
    ("Membrane", "membrane", "Membrane", [255, 0, 255]),
    ("CD45", "cd45", "CD45", [255, 255, 0]),
]
IMAGE_INFO = [{"name": n, "button_name": b, "color": c} for (_, n, b, c) in CHANNELS]


def step(msg):
    print(f"\n{'='*12} {msg} {'='*12}", flush=True)


# 1. identity transform (coords already in image px)
step("identity transform")
transform_path = DEGA / "micron_to_image_transform.csv"
import pandas as pd
M = np.diag([COORD_SCALE, COORD_SCALE, 1.0])
pd.DataFrame(M).to_csv(transform_path, sep=" ", header=False, index=False)

# 2. cell metadata in image (level-1) px
step("make_meta_cell_image_coord")
meta_cell_path = DEGA / "cell_metadata.parquet"
dega.pre.make_meta_cell_image_coord(
    TECH, str(transform_path), str(DATA_DIR / "cells.csv.gz"),
    str(meta_cell_path), image_scale=IMAGE_SCALE,
)

# 3. read CBG (cache to h5ad: the 550M-nnz WTx MTX takes ~10min to parse from text;
#    h5ad reloads in seconds for reruns)
step("read CBG")
import anndata as ad
import scipy.sparse as sp
h5ad_path = DATA_DIR / "cbg.h5ad"
if h5ad_path.exists():
    adata = ad.read_h5ad(h5ad_path)
    print("loaded cbg.h5ad", adata.shape)
else:
    cbg0 = dega.pre.read_cbg_mtx(str(DATA_DIR / "cell_feature_matrix"), technology=TECH)
    obs_idx = pd.Index(np.asarray(cbg0.index, dtype=str))
    var_idx = pd.Index(np.asarray(cbg0.columns, dtype=str))
    adata = ad.AnnData(
        X=sp.csr_matrix(cbg0.sparse.to_coo()).astype(np.float32),
        obs=pd.DataFrame(index=obs_idx),
        var=pd.DataFrame(index=var_idx),
    )
    adata.obs.index.name = None
    adata.var.index.name = None
    adata.write_h5ad(h5ad_path)
    del cbg0
    print("built+cached cbg.h5ad", adata.shape)
# pandas sparse cbg DataFrame for the celldega functions
cbg = pd.DataFrame.sparse.from_spmatrix(adata.X, index=adata.obs_names, columns=adata.var_names)
print("cbg:", cbg.shape)
# guard against corrupt cache: cell barcodes must look like c_<slide>_<fov>_<id>
assert str(cbg.index[0]).startswith("c_"), f"bad cbg index (corrupt h5ad?): {cbg.index[:3].tolist()}"
assert str(cbg.columns[0]).isalnum() or str(cbg.columns[0])[0].isalpha(), "bad cbg columns"

# 4. clustering (scanpy leiden) -> Xenium-style clusters.csv (CosMx has no 10x graphclust)
step("clustering -> clusters.csv")
clusters_csv = DATA_DIR / "analysis" / "clustering" / "gene_expression_graphclust" / "clusters.csv"
if not clusters_csv.exists():
    import scanpy as sc
    cl = adata.copy()
    sc.pp.filter_cells(cl, min_counts=20)
    sc.pp.normalize_total(cl)
    sc.pp.log1p(cl)
    sc.pp.highly_variable_genes(cl, n_top_genes=2000)
    cl = cl[:, cl.var.highly_variable].copy()
    sc.pp.scale(cl, max_value=10)
    sc.tl.pca(cl, n_comps=50)
    sc.pp.neighbors(cl, n_neighbors=15, n_pcs=50)
    sc.tl.leiden(cl, resolution=1.0, flavor="igraph", n_iterations=2, directed=False)
    print("n clusters:", cl.obs["leiden"].nunique())
    lab = (cl.obs["leiden"].astype(int) + 1).astype(str)
    df = pd.DataFrame({"Barcode": lab.index, "Cluster": lab.values})
    allc = pd.read_csv(DATA_DIR / "cells.csv.gz", usecols=["cell_id"])["cell_id"].astype(str)
    miss = set(allc) - set(df["Barcode"])
    if miss:
        df = pd.concat([df, pd.DataFrame({"Barcode": list(miss), "Cluster": "1"})], ignore_index=True)
    clusters_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(clusters_csv, index=False)
    del cl
else:
    print("clusters.csv exists, skip")

# 5. meta gene
step("make_meta_gene")
dega.pre.make_meta_gene(cbg, str(DEGA / "meta_gene.parquet"))

# 6. cluster gene expression (df_sig)
step("cluster_gene_expression")
dega.pre.cluster_gene_expression(TECH, str(DEGA), cbg, str(DATA_DIR))

# 6. CBG gene parquets (row groups)  -- skip + reconstruct chunk_info if already written
step("save_cbg_gene_parquets_row_groups")
cbg_dir = DEGA / "cbg"
cbg_files = sorted(cbg_dir.glob("chunk_*.parquet"), key=lambda f: int(f.stem.split("_")[1])) if cbg_dir.exists() else []
if cbg_files:
    import pyarrow.parquet as pq
    meta = pq.read_schema(str(cbg_files[0])).metadata or {}
    g2rg = json.loads(meta[b"gene_to_row_group"].decode()) if b"gene_to_row_group" in meta else {}
    cbg_chunk_info = {
        "directory": "cbg",
        "files": [f.name for f in cbg_files],
        "max_row_groups_per_file": int(meta.get(b"max_row_groups_per_file", str(MRG).encode())),
        "total_row_groups": int(meta.get(b"num_genes", str(len(g2rg)).encode())),
        "gene_to_row_group": g2rg,
    }
    print(f"cbg/ exists ({len(cbg_files)} files, {len(g2rg)} genes) -> reusing, skip rebuild")
else:
    cbg_chunk_info = dega.pre.save_cbg_gene_parquets_row_groups(
        TECH, str(DEGA), cbg, verbose=True, max_row_groups_per_file=MRG
    )

# 7. clusters + meta_cluster
step("create_cluster_and_meta_cluster")
dega.pre.create_cluster_and_meta_cluster(TECH, str(DEGA), str(DATA_DIR))


# 8. image pyramids from zarr
def build_image_pyramids():
    step("image pyramids from Napari zarr (level %s)" % ZARR_LEVEL)
    pyr = DEGA / "pyramid_images"
    pyr.mkdir(exist_ok=True)
    store = zarr.ZipStore(ZIP, mode="r")
    g = zarr.open_group(store=store, path=ZPFX, mode="r")
    image_tile_info = {}
    import shutil
    for cosmx_ch, name, _btn, _col in CHANNELS:
        t = time.time()
        # clean any prior (possibly partial) outputs so we always rebuild + populate info
        for p in (pyr / name, pyr / f"{name}_files"):
            if p.exists():
                shutil.rmtree(p)
        if (pyr / f"{name}.dzi").exists():
            (pyr / f"{name}.dzi").unlink()
        arr = g[cosmx_ch][ZARR_LEVEL][:]  # uint16 (y, x)
        # frugal contrast: percentile from a strided subsample (avoids a full-size copy)
        samp = arr[::4, ::4].ravel()
        samp = samp[samp > 0]
        lo, hi = (np.percentile(samp, (1, 99.5)) if samp.size else (0, 1))
        del samp
        hi = float(max(hi, lo + 1))
        lo = float(lo)
        # in-place clip on the uint16 array, then scale into uint8 (one temp at a time)
        np.clip(arr, lo, hi, out=arr)
        arr -= np.uint16(lo)
        arr8 = (arr * (255.0 / (hi - lo))).astype(np.uint8)
        del arr
        arr8 = np.ascontiguousarray(arr8)
        h, w = arr8.shape
        vi = pyvips.Image.new_from_memory(arr8.data, w, h, 1, "uchar")
        vi.dzsave(str(pyr / name), tile_size=512, overlap=0, suffix=".webp[Q=90]")
        del arr8, vi
        print(f"  {name}: {w}x{h} dzsave {time.time()-t:.0f}s", flush=True)
        # pack tiles -> parquet, delete source tiles
        info = dega.pre.pack_image_tiles_to_parquet(
            str(pyr), name, str(pyr / name), image_format=".webp",
            delete_source_tiles=True,
        )
        image_tile_info[name] = info
    store.close()
    return image_tile_info


# free the cell-by-gene matrix before the memory-heavy image read (avoids OOM)
import gc
del cbg, adata
gc.collect()

image_tile_info = build_image_pyramids()

# 9. transcript tiles (row groups)
# NOTE: make_trx_tiles_row_groups loads the whole transcript parquet into memory. The full WTx
# tx file has ~1.12B transcripts (~60GB in RAM) which would OOM, so we use a subsampled parquet
# (~250M transcripts) here. Full-resolution would require a streaming enhancement to the pre module.
step("make_trx_tiles_row_groups")
import re as _re
import pyarrow.parquet as _pq
trx_out = DEGA / "transcripts"
trx_files = sorted(trx_out.glob("chunk_*.parquet"), key=lambda f: int(f.stem.split("_")[1])) if trx_out.exists() else []
if trx_files:
    # reuse existing transcript tiles; reconstruct grid/bounds/chunk_info from chunk metadata
    _m = _pq.read_schema(str(trx_files[0])).metadata or {}
    tile_grid_info = json.loads(_m[b"tile_grid_info"].decode())
    tile_bounds = {k: tile_grid_info[k] for k in ("x_min", "y_min", "x_max", "y_max")}
    trx_chunk_info = {
        "directory": "transcripts",
        "files": [f.name for f in trx_files],
        "max_row_groups_per_file": int(_m.get(b"max_row_groups_per_file", str(MRG).encode())),
        "total_row_groups": sum(_pq.ParquetFile(str(f)).num_row_groups for f in trx_files),
    }
    print(f"transcripts/ exists ({len(trx_files)} files) -> reusing, skip tiling")
else:
    trx_path = DATA_DIR / "transcripts_sub.parquet"
    if not trx_path.exists():
        trx_path = DATA_DIR / "transcripts.parquet"
    # Reconcile gene names: CosMx exprMat (-> meta_gene) and tx file disagree on a few separators
    # (e.g. exprMat 'C4B-2' vs tx 'C4B_2'). Remap tx feature_name to the meta_gene name via a
    # separator-insensitive canonical key so make_trx_tiles' gene->int mapping has no nulls.
    _mg = set(pd.read_parquet(DEGA / "meta_gene.parquet").index.astype(str))
    _txg = pl.scan_parquet(trx_path).select("feature_name").unique().collect()["feature_name"].to_list()
    _canon = lambda s: _re.sub(r"[-_.]", "", str(s)).upper()
    _cmap = {}
    for _g in _mg:
        _cmap.setdefault(_canon(_g), _g)
    _rename = {g: _cmap[_canon(g)] for g in _txg if g not in _mg and _canon(g) in _cmap}
    _unmapped = [g for g in _txg if g not in _mg and _canon(g) not in _cmap]
    if _unmapped:
        print(f"WARNING: {len(_unmapped)} tx targets have no meta_gene match (dropping): {_unmapped[:10]}")
    if _rename or _unmapped:
        fixed = DATA_DIR / "transcripts_sub_mapped.parquet"
        lf = pl.scan_parquet(trx_path)
        if _rename:
            lf = lf.with_columns(pl.col("feature_name").replace(_rename))
        if _unmapped:
            lf = lf.filter(~pl.col("feature_name").is_in(_unmapped))
        lf.sink_parquet(fixed, compression="zstd")
        trx_path = fixed
        print(f"remapped {len(_rename)} gene names -> {trx_path}")
    tile_bounds, tile_grid_info, trx_chunk_info = dega.pre.make_trx_tiles_row_groups(
        TECH, str(trx_path), str(transform_path),
        str(DEGA / "transcripts"), coarse_tile_factor=10, tile_size=TILE_SIZE,
        chunk_size=100000, verbose=False, image_scale=IMAGE_SCALE, max_workers=MAX_WORKERS,
        path_dega_files=str(DEGA), max_row_groups_per_file=MRG,
    )
print("tile_bounds:", tile_bounds)

# 10. cell boundary tiles (row groups)
step("make_cell_boundary_tiles_row_groups")
cell_chunk_info = dega.pre.make_cell_boundary_tiles_row_groups(
    TECH, str(DATA_DIR / "cell_boundaries.parquet"), str(DEGA / "cell_segmentation"),
    str(DATA_DIR / "cells.csv.gz"), str(transform_path), coarse_tile_factor=10,
    tile_size=TILE_SIZE, tile_bounds=tile_bounds, image_scale=IMAGE_SCALE,
    max_workers=MAX_WORKERS, path_dega_files=str(DEGA), max_row_groups_per_file=MRG,
)

# 11. landscape parameters
step("save_landscape_parameters")
dega.pre.save_landscape_parameters(
    TECH, str(DEGA), image_name="dapi_files", tile_size=TILE_SIZE,
    image_info=IMAGE_INFO, image_format=".webp", use_int_index=True,
    use_row_groups=True, tile_grid_info=tile_grid_info, image_tile_info=image_tile_info,
    trx_chunk_info=trx_chunk_info, cell_chunk_info=cell_chunk_info, cbg_chunk_info=cbg_chunk_info,
)
print("\nDONE. DegaFiles at", DEGA)
print(json.dumps(json.load(open(DEGA / "landscape_parameters.json")), indent=2)[:1500])


## Step 5 — copy DegaFiles to the workbench
DegaFiles were written to fast local disk. Copy to the shared (s3fs) workbench when verified.

In [ ]:
# import shutil
# shutil.copytree('/home/jovyan/local/cosmx_dega_files/S0',
#                 '/home/jovyan/workbench/CosMx_Celldega/cosmx_dega_files/S0')
print('Uncomment to copy. DegaFiles: /home/jovyan/local/cosmx_dega_files/S0')